# 조3조 기말 프로젝트 — 지역·기온별 전력 패턴 분석

**과목:** 파이썬데이터분석 | **교수:** 서지훈 | **조:** 3조

하유빈 · 곽소민 · 황준연 · 박선준 · 공지수

---
이 노트북은 `data/` 샘플 CSV만으로 실행됩니다.  
전체 코드·데이터는 GitHub `granite-tsfm` 레포 참고.

## 1. 데이터 로드

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

BASE = Path('.').resolve()  # 00조_기말 폴더에서 실행
DATA = BASE / 'data'
RESULTS = BASE / 'results'

plt.rcParams['font.family'] = ['AppleGothic', 'Apple SD Gothic Neo', 'Malgun Gothic', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False

raw = pd.read_csv(DATA / 'sample_01_raw_kpx.csv', parse_dates=['date'])
merged = pd.read_csv(DATA / 'sample_02_merged_hourly.csv', parse_dates=['ts'])
district = pd.read_csv(DATA / 'sample_03_seoul_district_monthly.csv')

print('=== RAW (KPX 원본 샘플) ===')
display(raw.head())
print(f'행 수: {len(raw):,} | 지역: {raw.region.unique().tolist()}')

print('\n=== MERGED (정제·병합 샘플) ===')
display(merged.head())
print(f'행 수: {len(merged):,}')

print('\n=== 서울 구별 월별 에너지 ===')
display(district.head())
print(f'행 수: {len(district):,} | 구: {district.district.nunique()}개')

## 2. Track 1 — 4개 지역 시간대별 패턴 (샘플 1주일)

같은 전력망 안에서도 **서울(업무시간)** vs **강원(상대적 안정)** 패턴이 다릅니다.

In [ ]:
hourly = merged.groupby(['region', 'hour'])['power'].mean().reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
colors = {'서울시': '#0ea5e9', '부산시': '#f97316', '대전시': '#22c55e', '강원도': '#a78bfa'}
for region in hourly['region'].unique():
    sub = hourly[hourly['region'] == region]
    ax.plot(sub['hour'], sub['power'], marker='o', label=region, color=colors.get(region, 'gray'))
ax.set_xlabel('시간 (0~23시)')
ax.set_ylabel('평균 전력거래량 (MWh)')
ax.set_title('4개 지역 시간대별 평균 거래량 (샘플: 2024-01-01~07)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Track 1 — 기온 vs 전력 (U자 확인, 샘플)

기온과 전력의 관계는 **직선이 아니라 U자**에 가깝습니다.

In [ ]:
seoul = merged[merged['region'] == '서울시']
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(seoul['temp'], seoul['power'], alpha=0.4, s=15, c='#0ea5e9')
ax.set_xlabel('기온 (°C)')
ax.set_ylabel('전력거래량 (MWh)')
ax.set_title('서울 — 기온 × 전력거래량 (샘플 1주일)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Track 2 — 서울 25개 구 연간 사용량 순위

In [ ]:
annual = district.groupby('district')['usage'].sum().sort_values(ascending=True)
top5 = annual.tail(5)
bottom5 = annual.head(5)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].barh(top5.index, top5.values / 1e6, color='#f97316')
axes[0].set_title('사용량 상위 5개 구 (백만 MWh)')
axes[1].barh(bottom5.index, bottom5.values / 1e6, color='#0ea5e9')
axes[1].set_title('사용량 하위 5개 구 (백만 MWh)')
plt.suptitle(f'강남 ÷ 강북 ≈ {annual.iloc[-1]/annual.iloc[0]:.1f}배 (전체 기간 합)', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Track 3 — 2024 모델 검증 결과 (재학습 없음)

- **학습:** 2020~2023 서울 데이터  
- **시험:** 2024년 전체 (한 번도 안 본 데이터)  
- **방법 A:** 4일치 한 번에 예측 → 평균 오차 **약 111 MWh (17%)**  
- **방법 B:** 하루씩 이어 맞추기 → 평균 오차 **약 159 MWh** → **약 30% 더 큼**

In [ ]:
with open(RESULTS / 'backtest_summary_2020_2024.json', encoding='utf-8') as f:
    backtest = json.load(f)

y2024 = backtest['years']['2024']
print('=== 2024년 out-of-sample 검증 (canonical) ===')
print(f"4일 한 번에 (TTM):     {y2024['ttm']} MWh")
print(f"하루씩 이어 (GRU):      {y2024['gru']} MWh")
print(f"전문AI+서울맞춤:       {y2024['hybrid']} MWh")
print(f"작년 같은 시간 기준:  {y2024['naive']} MWh")
print()
improve = (y2024['gru'] - y2024['ttm']) / y2024['gru'] * 100
print(f"→ 4일 한 번에가 하루씩 이어보다 평균 오차 {improve:.0f}% 적음")

labels = ['4일 한 번에', '하루씩 이어', '전문AI+서울맞춤', '작년같은시간']
vals = [y2024['ttm'], y2024['gru'], y2024['hybrid'], y2024['naive']]
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(labels, vals, color=['#2563eb', '#f97316', '#7c3aed', '#9ca3af'])
ax.set_ylabel('평균 오차 (MWh) — 작을수록 좋음')
ax.set_title('2024년 1년 검증 — 예측 방법 비교')
for i, v in enumerate(vals):
    ax.text(i, v + 3, f'{v}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. 결론

1. **지역마다** 전력 거래 구조가 다르다 (발전소 유무 → 충남 vs 서울 20배).  
2. **서울 안에서도** 구마다 에너지 사용이 5배 이상 차이난다.  
3. **2024 시험**에서 4일 앞 예측이 "하루씩 이어 맞추기"보다 평균 **30% 덜 틀렸다**.  
4. 같은 관리·같은 모델로는 안 되고, **지역·구별 맞춤**이 필요하다.

---
**전체 코드:** GitHub granite-tsfm | **모델 가중치:** `models/hybrid_seoul.pt`